# 14 · Corpus80 parça 0'ı sıkıştırılmış vaka dosyalarına çevir (Kaggle, CPU)

**Neden?** Raptor-80'i yerel RTX 5090 sunucuda eğitmek istiyoruz ama sunucuda ~36 GB boş disk var;
ham corpus 40 GB. Bu notebook parçanın her vakasını ayrı bir sıkıştırılmış `.npz` dosyasına yazar
(dosya adı = birleşik corpus'taki satır numarası). Sunucu yalnızca bu sıkıştırılmış dosyaları indirir
ve eğitim script'i `RAPTOR_NPZ_DIR` ile bunları okur (`third_party/raptor/train_knee.py`).

In [ ]:
import os, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np

SHARD = 0
def find(name):
    for d, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [x for x in dirs if x not in ("train_series", "test_series")]
        if name in files:
            return Path(d) / name
    raise FileNotFoundError(name)

vols = np.load(find(f"corpus_vols_s{SHARD:02d}.npy"), mmap_mode="r")
masks = np.load(find(f"corpus_masks_s{SHARD:02d}.npy"))
ids = np.load(find(f"corpus_ids_s{SHARD:02d}.npy"), allow_pickle=True)
OFFSET = SHARD * 1469            # parçalar 1469'ar vaka (4.407 / 3); satır = OFFSET + i
OUT = Path("/kaggle/working/npz"); OUT.mkdir(parents=True, exist_ok=True)
print(vols.shape, OFFSET)

t0 = time.time()
def save(i):
    np.savez_compressed(OUT / f"{OFFSET + i:05d}.npz", v=np.asarray(vols[i]))
with ThreadPoolExecutor(8) as ex:
    list(ex.map(save, range(len(vols))))
np.save(f"/kaggle/working/masks_s{SHARD:02d}.npy", masks)
np.save(f"/kaggle/working/ids_s{SHARD:02d}.npy", ids, allow_pickle=True)
size = sum(f.stat().st_size for f in OUT.glob("*.npz")) / 1e9
print(f"{len(vols)} vaka -> {size:.1f} GB sıkıştırılmış ({size / (vols.nbytes / 1e9):.0%}), {(time.time() - t0) / 60:.1f} dk")